# LeiMai Oracle / Colab Matrix Runner (Fallback)

Use this notebook when Kaggle is unavailable.

In [ ]:
REPO_URL = "https://github.com/Le1Ma1/leimai-oracle.git"
BRANCH = "main"
BATCH_INDEX = 1
BATCH_TOTAL = 3
MAX_ROUNDS = 1
RUN_VALIDATE = True
SYMBOLS = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ADAUSDT",
    "DOGEUSDT", "LTCUSDT", "LINKUSDT", "BCHUSDT", "TRXUSDT",
    "ETCUSDT", "XLMUSDT", "EOSUSDT", "XMRUSDT", "ATOMUSDT",
]


In [ ]:
from pathlib import Path
import os
import subprocess

repo_root = Path('/content/leimai-oracle')
if not repo_root.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo_root)], check=True)
os.chdir(repo_root)
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-r', 'engine/requirements.txt'], check=True)
print('repo_root=', repo_root)


In [ ]:
def split_batches(symbols, total):
    total = max(1, int(total))
    n = len(symbols)
    out = []
    for i in range(total):
        s = round(i * n / total)
        e = round((i + 1) * n / total)
        bucket = symbols[int(s):int(e)]
        if bucket:
            out.append(bucket)
    return out or [symbols]

batches = split_batches(SYMBOLS, BATCH_TOTAL)
batch_idx = max(1, min(BATCH_INDEX, len(batches)))
batch_symbols = batches[batch_idx - 1]
print('batch', batch_idx, '/', len(batches), batch_symbols)

env = os.environ.copy()
env['ENGINE_UNIVERSE_SYMBOLS'] = ','.join(batch_symbols)
env['ENGINE_RULE_ENGINE_MODE'] = 'feature_native'
env['ENGINE_OPTIMIZATION_GATE_MODES'] = 'gated,ungated'
env['ENGINE_OPTIMIZATION_TIMEFRAMES'] = '1m'
env['ENGINE_OPTIMIZATION_WINDOWS'] = 'all,30d,90d,360d'


In [ ]:
subprocess.run(['python', '-m', 'engine.src.main', '--mode', 'iterate', '--max-rounds', str(MAX_ROUNDS)], env=env, check=True)
if RUN_VALIDATE:
    subprocess.run(['python', '-m', 'engine.src.main', '--mode', 'validate'], env=env, check=True)
subprocess.run([
    'python', 'scripts/cloud_dispatch.py', 'manifest',
    '--target', 'colab',
    '--status', 'completed',
    '--batch-index', str(batch_idx),
    '--batch-total', str(len(batches)),
    '--symbols', ','.join(batch_symbols),
    '--tasks-total', '1',
    '--tasks-done', '1',
    '--mirror-to-daily',
], check=True)
print('manifest written to engine/artifacts/cloud/cloud_run_manifest.json')
